In this layer, we will prepare the charts to see the distribution of the data across the state. 

**First, map the MSSA borders**

the dataset is designed to map the GEOID of census tracts. MSSAs are comprised of one or more census tracts

In [0]:
# the .csv file of mssa_geo table is saved here - https://drive.google.com/file/d/1Ns2pCxnJNW7NaQmD6gYFV8B0DuRviaCK/view?usp=drive_link

mssa_geo = spark.read.table("ca_healthcare_fac_bronze.mssa_data_bronze.mssa_geo").toPandas()
mssa_geo.head()

In [0]:
from shapely.geometry import Polygon

def convert_polygon(rings):
    return Polygon(rings[0])


In [0]:
mssa_geo['geometry_rings'] = mssa_geo['geometry_rings'].apply(convert_polygon)
# mssa_geo.head()

In [0]:
import geopandas as gpd

mssa_gpd = gpd.GeoDataFrame(mssa_geo, geometry='geometry_rings', crs="EPSG:4326")
mssa_gpd.head()

In [0]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(16, 20))

mssa_gpd.plot(
    ax=ax,
    facecolor='lightblue',
    edgecolor='black',
    linewidth=0.3,
)

ax.set_xlim(-124.5, -113.5)
ax.set_ylim(32.0, 42.5)

ax.set_title('California Census Tracts', fontsize=16)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

plt.tight_layout()
plt.show()  # renders inline in Databricks notebook

**CDC Places data**
Import, convert to gpd, then map

In [0]:
cdc_silver = spark.read.table("ca_healthcare_fac_silver.cdc_places_silver.cdc_places_mssa_v0").toPandas() 
#  cdc_places_mssa table is saved in .csv file here - https://drive.google.com/file/d/1fi3P_KCF9PfPQw9Z3cYiXKuS8Z4Ymtb4/view?usp=drive_link


In [0]:
cdc_silver_w_rings = cdc_silver.merge(mssa_geo[['GEOID', 'geometry_rings']], left_on='LocationName',right_on='GEOID', how='left')

In [0]:
# May not need to convert polygon here, but it's a good idea to do it once before plotting
cdc_silver_w_rings['geometry_rings'] = cdc_silver_w_rings['geometry_rings'].apply(convert_polygon)

In [0]:
import geopandas as gpd
cdc_gpd = gpd.GeoDataFrame(cdc_silver_w_rings, geometry=cdc_silver_w_rings['geometry_rings'], crs="EPSG:4326")

In [0]:
measure_values = sorted(cdc_gpd['Measure'].dropna().unique())
# measure_values

Create separate maps for measures

In [0]:
import numpy as np

# Simplify geometry ONCE before any plotting ───────────────────────

cdc_gpd['geometry'] = cdc_gpd['geometry'].simplify(
    tolerance=0.005,
    preserve_topology=True
)

In [0]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# ── Plot one heatmap per measure ─────────────────────────────────────
for measure in measure_values:

    # Filter for this measure
    measure_gdf = cdc_gpd[cdc_gpd['Measure'] == measure].copy()

    # Skip if no valid data
    if measure_gdf['Data_Value'].dropna().empty:
        print(f"Skipping '{measure}' — no Data_Value entries")
        continue

    vmin = measure_gdf['Data_Value'].quantile(0.05)
    vmax = measure_gdf['Data_Value'].quantile(0.95)

    fig, ax = plt.subplots(1, 1, figsize=(12, 14))

    # Plot census tract heatmap for Data_Value
    measure_gdf.plot(
        ax=ax,
        column='Data_Value',
        cmap='YlOrRd',
        edgecolor='none',
        linewidth=0,
        legend=False,
        vmin=vmin,
        vmax=vmax,
        missing_kwds={'color': 'lightgrey', 'label': 'No Data'}
    )

    # Manually add colorbar (avoids Legend.__init__ conflict from before)
    sm = cm.ScalarMappable(
        cmap='YlOrRd',
        norm=mcolors.Normalize(vmin=vmin, vmax=vmax)
    )
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation='horizontal', pad=0.02, shrink=0.5)
    cbar.set_label('Data Value', fontsize=10)

    # Map extent — California bounds
    ax.set_xlim(-124.5, -113.5)
    ax.set_ylim(32.0, 42.5)
    ax.axis('off')

    # Annotate with data coverage
    coverage = measure_gdf['Data_Value'].notna().sum()
    total    = len(measure_gdf)

    ax.set_title(
        f'{measure} - California Census Tracts',
        fontsize=13,
        fontweight='bold',
        pad=12
    )

    plt.tight_layout()
    plt.show()
    plt.close(fig)   

print("\nAll measures complete!")

**Processing Census data**

Gather and map each numeric column in the census table

In [0]:
# the .csv version of the census_w_mssa table is available here - https://drive.google.com/file/d/1vVUZHNGZ2_Yr7_9WGRVvhV4Au-9YtfeT/view?usp=drive_link

census = spark.read.table('ca_healthcare_fac_silver.census_data_silver.census_w_mssa').toPandas()
census.head()

In [0]:
col_types = {
              'Population': 'int', 
              'Avg_household_size': 'float', 
              'Households_w_ppl_under_18_y': 'float',
              'Households_w_ppl_60_plus_y': 'float', 
              'Households_w_ppl_65_plus_y': 'float',
              'Householder_living_alone': 'float', 
              'Lonely_Householder_65_plus_y': 'float',
              'Car_truck_van': 'float', 'Drive_alone': 'float', 
              'carpooled': 'float', 'Pub_transport': 'float', 
              'walked': 'float', 'bicycle_2_work': 'float', 
              'Taxi_ride_hail_motorbike_other': 'float', 
              'Worked_from_home': 'float', 
              'pct_below_poverty': 'float', 
              'household_w_grand_parents_children': 'float',
              'avg_weekly_hr_worked_per_worker': 'float', 
              'pct_occ_household_1_person': 'float',
              'pct_occ_household_2_person': 'float', 
              'pct_occ_household_3_person': 'float',
              'pct_occ_household_4_n_up': 'float', 
              'median_household_income': 'int',
              'pct_health_insured': 'float', 
              'pct_pbl_health_insured': 'float'
       }

In [0]:
census = census.astype(col_types)
# census.info()


In [0]:
from shapely.geometry import Polygon
import geopandas as gpd

census['geometry_rings'] = census['geometry_rings'].apply(convert_polygon)


In [0]:
census_gpd = gpd.GeoDataFrame(census, geometry='geometry_rings', crs="EPSG:4326")
census_gpd.head()

Map the census data

In [0]:
metrics = {
    'GEO_ID': 'Census_GEO_ID',
    'NAME': 'Census_Tract_Name',
    'Population': 'Tract Population',
    'Avg_household_size': 'Average Household size', 
    'Households_w_ppl_under_18_y': 'Percent of households with people < 18 years of age',
    'Households_w_ppl_60_plus_y': 'Percent of Households with people 60+ years of age',
    'Households_w_ppl_65_plus_y': 'Percent of Households with people 65+ years of age',
    'Householder_living_alone': 'Percent of Householder LIVING ALONE',
    'Lonely_Householder_65_plus_y': 'Percent of householder living alone are 65+ years of age',
    'Car_truck_van': 'Percent of population own Car Truck or Van',
    'Drive_alone': 'Percent of population Drive alone to work',
    'carpooled': 'Percent of population Carpooled to work',
    'Pub_transport': 'Percent of population commute to work by Public Transport',
    'walked': 'Percent of population walked to work',
    'bicycle_2_work': 'Percent of population ride bicycle to work', 
    'Taxi_ride_hail_motorbike_other': 'Percent of population commute to work by \n Taxi, Ride Hail, or Motorbike',
    'Worked_from_home': 'Percent of population work from home',
    'pct_below_poverty': 'Percent of population below poverty level', 
    'household_w_grand_parents_children': 'Percent of household with grand parents and children',
    'avg_weekly_hr_worked_per_worker': 'avg_weekly_hr_worked_per_worker',
    'pct_occ_household_1_person': 'Percent of households occupied by 1 person',
    'pct_occ_household_2_person': 'Percent of households occupied by 2 person', 
    'pct_occ_household_3_person': 'Percent of households occupied by 3 person',
    'pct_occ_household_4_n_up': 'Percent of households occupied by 4 or More people',
    'median_household_income': 'Median household income',
    'pct_health_insured': 'Population percentage with health insurance', 
    'pct_pbl_health_insured': 'Population percentage with public health insurance plans',
    'MSSAID': 'MSSAID',
    'MSSANM': 'MSSANM'
}

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# Simplify geometry ONCE before any plotting ───────────────────────
census_gpd_plot = census_gpd.copy()
census_gpd_plot['geometry_rings'] = census_gpd_plot['geometry_rings'].simplify(
    tolerance=0.005,
    preserve_topology=True
)

# Only plot numeric metrics — skip ID/name columns ─────────────────
non_numeric = ['GEO_ID', 'NAME', 'MSSAID', 'MSSANM']

numeric_metrics = {
    col: label 
    for col, label in metrics.items() 
    if col not in non_numeric
}

print(f"Plotting {len(numeric_metrics)} numeric metrics...")

# Plot one figure per metric — saves memory, shows output sooner ───
for col, label in numeric_metrics.items():

    fig, ax = plt.subplots(1, 1, figsize=(12, 14))

    vmin = census_gpd_plot[col].quantile(0.05)
    vmax = census_gpd_plot[col].quantile(0.95)

    census_gpd_plot.plot(
        ax=ax,
        column=col,
        cmap='YlOrRd',
        edgecolor='none',        # removing edges speeds up rendering a lot
        linewidth=0,
        legend=False,
        vmin = vmin,
        vmax = vmax,
        missing_kwds={'color': 'lightgrey'}
    )

    import matplotlib.cm as cm
    import matplotlib.colors as mcolors

    sm = cm.ScalarMappable(
        cmap='YlOrRd',
        norm=mcolors.Normalize(vmin=vmin, vmax=vmax)
    )
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation='horizontal',
        pad=0.02,
        shrink=0.4
    )
    cbar.set_label(label, fontsize=10)

    ax.set_xlim(-124.5, -113.5)
    ax.set_ylim(32.0, 42.5)
    ax.set_title(f'CA Census Tracts \n{label}', fontsize=14, fontweight='bold')
    ax.axis('off')

    plt.tight_layout()
    plt.show()

print(f"Done printing all labels")

**CA Land Data**
Calculate total open space and public lands in each GEOID and map 

In [0]:
# ca_public_land_mssa table can be accessed in .csv file here - https://drive.google.com/file/d/11nB8K13V7r-vldBLokKziJL59reW0KGi/view?usp=drive_link

ca_land_gold = spark.sql("""
    SELECT -- might need to get separate count by GEOID
        plm.GEOID, plm.MSSAID, plm.MSSANM,
        sum(plm.Shape__Area) AS total_open_space_public_land
    FROM ca_healthcare_fac_silver.ca_public_land_mssa_silver.ca_public_land_mssa as plm
    GROUP BY plm.GEOID, plm.MSSAID, plm.MSSANM
    ORDER BY plm.MSSAID ASC;""")

ca_land_pddf = ca_land_gold.toPandas()



In [0]:
ca_land_geo = ca_land_pddf.merge(mssa_gpd[['GEOID', 'geometry_rings']], on='GEOID', how='inner')

In [0]:
ca_land_gpd = gpd.GeoDataFrame(ca_land_geo, geometry='geometry_rings', crs="EPSG:4326")
ca_land_gpd.head(10)

In [0]:
vmin = ca_land_gpd['total_open_space_public_land'].quantile(0.05)
vmax = ca_land_gpd['total_open_space_public_land'].quantile(0.95)

fig, ax = plt.subplots(1, 1, figsize=(12, 14))

# Plot census tract polygons colored by Data_Value
ca_land_gpd.plot(
    ax=ax,
    column='total_open_space_public_land',
    cmap='YlOrRd',
    edgecolor='none',
    linewidth=0,
    legend=False,
    vmin=vmin,
    vmax=vmax,
    missing_kwds={'color': 'lightgrey', 'label': 'No Data'}
)

# Manually add colorbar (avoids Legend.__init__ conflict from before)
sm = cm.ScalarMappable(
    cmap='YlOrRd',
    norm=mcolors.Normalize(vmin=vmin, vmax=vmax)
)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='horizontal', pad=0.02, shrink=0.5)
cbar.set_label('Total Open Space & Public Land', fontsize=10)

# Map extent — California bounds
ax.set_xlim(-124.5, -113.5)
ax.set_ylim(32.0, 42.5)
ax.axis('off')

filepath  = f'/tmp/cdc_{safe_name}.png'

plt.tight_layout()
plt.savefig(filepath, dpi=100, bbox_inches='tight')
plt.show()



**Processing HPSA silver data.** 

calculate the site counts by GEOIDs

In [0]:
# The .csv version of hpsa_ca_dashboard_mssa table can be accessed here - https://drive.google.com/file/d/1NFFJBbARJ-1dvpIuB5rwvQqSZRst1Szl/view?usp=drive_link


hpsa_site_counts = spark.sql("""SELECT 
    hpsa.GEOID,hpsa.site_type,
    hpsa.longitude, hpsa.latitude, 
    count(hpsa.site_dashboard_surrogate_key) AS hpsa_site_cnt
FROM ca_healthcare_fac_silver.hpsa_dashboard_silver.hpsa_ca_dashboard_mssa as hpsa
WHERE hpsa.MSSAID IS NOT NULL
GROUP BY hpsa.GEOID,hpsa.site_type,
        hpsa.longitude, hpsa.latitude;""")

hpsa_site_counts_pddf = hpsa_site_counts.toPandas()


In [0]:
hpsa_site_cnt_gdf = hpsa_site_counts_pddf.merge(mssa_gpd[['GEOID', 'geometry_rings']], on='GEOID', how='inner')


In [0]:
import geopandas as gpd

hpsa_site_cnt_gpd = gpd.GeoDataFrame(hpsa_site_cnt_gdf, geometry=hpsa_site_cnt_gdf.geometry_rings, crs="EPSG:4326")

In [0]:
import numpy as np

# Simplify geometry ONCE before any plotting ───────────────────────

hpsa_site_cnt_gpd['geometry'] = hpsa_site_cnt_gpd['geometry'].simplify(
    tolerance=0.005,
    preserve_topology=True
)

Create the table to count the total number of HPSA sites in the census Tract that will be merged to the individual site type counts

In [0]:
total_hpsa_counts = spark.sql(
    """SELECT 
    hpsa.GEOID, count(hpsa.site_dashboard_surrogate_key) AS total_hpsa_sites
    FROM ca_healthcare_fac_silver.hpsa_dashboard_silver.hpsa_ca_dashboard_mssa as hpsa
    GROUP BY hpsa.GEOID
    """).toPandas()
total_hpsa_counts.head()

In [0]:
hpsa_agg = hpsa_site_cnt_gpd.merge(total_hpsa_counts, on='GEOID', how='left')
hpsa_agg.head()

In [0]:
# to assign colors to site types
site_types = sorted(hpsa_agg['site_type'].unique())
site_types

Assign colors to each site type

In [0]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

color_palette = plt.cm.tab10.colors if len(site_types) <= 10 else plt.cm.tab20.colors
site_color_map = {stype: color_palette[i % len(color_palette)] for i, stype in enumerate(site_types)}

print("\nColor assignments:")
for stype, color in site_color_map.items():
    print(f"  {stype}: {color}")

In [0]:
# ── Plot the map ──────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(14, 18))

# Base layer — census tract boundaries (light grey fill)
mssa_gpd.plot(
    ax=ax,
    facecolor='#f0f0f0',
    edgecolor='#cccccc',
    linewidth=0.2
)

# ── Plot dots per site_type ───────────────────────────────────────────────────
# Scale dot size by total_sites in that census tract
# Adjust base_size and scale_factor to taste
base_size  = 20
scale_factor = 15

for stype in site_types:
    subset = hpsa_agg[hpsa_agg['site_type'] == stype].copy()

    # Drop rows with missing coordinates
    subset = subset.dropna(subset=['latitude', 'longitude'])

    # Dot size scales with total sites in that census tract
    dot_sizes = base_size + (subset['total_hpsa_sites'] * scale_factor)

    ax.scatter(
        x=subset['longitude'],
        y=subset['latitude'],
        s=dot_sizes,
        c=[site_color_map[stype]],
        alpha=0.75,
        edgecolors='white',
        linewidths=0.4,
        label=stype,
        zorder=3              # render dots above the base map
    )

# ── Annotate count labels on dots ────────────────────────────────────
# Show site count number on each dot where total_sites > 1
for _, row in hpsa_agg.dropna(subset=['latitude', 'longitude']).iterrows():
    if row['total_hpsa_sites'] > 1:
        ax.annotate(
            str(int(row['total_hpsa_sites'])),
            xy=(row['longitude'], row['latitude']),
            fontsize=5,
            ha='center',
            va='center',
            color='white',
            fontweight='bold',
            zorder=4
        )

# ── Legend and labels ─────────────────────────────────────────────────

# Color legend — one entry per site_type
color_legend_handles = [
    mpatches.Patch(color=site_color_map[stype], label=stype)
    for stype in site_types
]

# Size legend — shows what dot sizes mean
size_legend_handles = [
    ax.scatter([], [], s=base_size + (n * scale_factor),
               c='grey', alpha=0.75, label=f'{n} site(s)')
    for n in [1, 5, 10, 20]
]

# Add both legends
legend1 = ax.legend(
    handles=color_legend_handles,
    title='Site Type',
    loc='lower left',
    fontsize=8,
    title_fontsize=9,
    framealpha=0.9
)
ax.add_artist(legend1)   # keep first legend when adding second

legend2 = ax.legend(
    handles=size_legend_handles,
    title='Total Sites\nin Census Tract',
    loc='lower right',
    fontsize=8,
    title_fontsize=9,
    framealpha=0.9
)

ax.set_xlim(-124.5, -113.5)
ax.set_ylim(32.0, 42.5)
ax.axis('off')
ax.set_title(
    'California HPSA Sites by Type and Census Tract\n'
    '(dot color = site type, dot size = total sites in tract)',
    fontsize=14,
    fontweight='bold',
    pad=15
)

plt.tight_layout()
plt.savefig('/tmp/hpsa_sites_map.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print("✅ Map saved to /tmp/hpsa_sites_map.png")